# 04_corruption_benchmark.ipynb

Xem ảnh mẫu và hiệu chỉnh các mức suy giảm trên tập validation bằng reference model. Không train lại, không mở tập test. Notebook chạy trên Kaggle với GPU.

Các mức nằm ở `configs/corruption.yaml`. Sau khi xem kết quả ở đây, mức được chỉnh nếu cần rồi khóa trước khi chạm vào tập test.

## 1. Môi trường

Internet để On. Accelerator chọn GPU. Dataset input gồm `cthng123/durian-leaf-disease-dataset` và dataset chứa checkpoint của reference model.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "corruption-benchmark"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Checkpoint

Lấy checkpoint của reference model đã chọn ở task 3.3.

In [ ]:
import json

from src.evaluation.reference import collect_checkpoint

reference = json.loads((REPO_DIR / "reports/baselines/reference_selection.json").read_text(encoding="utf-8"))["selection"]["reference"]
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
checkpoint = collect_checkpoint("/kaggle/input", f"{reference}_seed42.pt", CHECKPOINT_DIR)
print(f"Reference model: {reference}")
print(f"Checkpoint: {checkpoint.stat().st_size / 1024**2:.1f} MB")

## 3. Ảnh mẫu

Một ảnh validation qua từng họ suy giảm. Cột đầu là ảnh sạch, các cột sau là mức 1, 2, 3.

In [ ]:
import yaml
from IPython.display import Image, display

from src.corruption import example_rows, load_conditions, load_seed
from src.evaluation import plot_corruption_examples

CONFIG = "configs/corruption.yaml"
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text(encoding="utf-8"))
conditions = load_conditions(CONFIG, "val")
rows = example_rows(pipeline_cfg, "val", conditions, load_seed(CONFIG), index=0)
figure_path = REPO_DIR / "reports/corruptions/severity_examples.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
plot_corruption_examples(rows, figure_path)
display(Image(filename=figure_path))

## 4. Hiệu chỉnh trên validation

Reference model được đánh giá trên ảnh sạch và 15 điều kiện. Một mức hợp lý làm điểm giảm dần theo mức và không đưa điểm về mức đoán bừa ngay từ mức nhẹ.

In [ ]:
import pandas as pd

from src.evaluation.robustness import run_calibration

record = run_calibration(CONFIG, CHECKPOINT_DIR)
table = pd.DataFrame(
    [
        {"family": name.rsplit("_s", 1)[0], "severity": int(name.rsplit("_s", 1)[1]), "value": entry["value"], "macro_f1": entry["macro_f1"]}
        for name, entry in record["conditions"].items()
    ]
)
display(table.pivot(index="family", columns="severity", values="macro_f1").assign(clean=record["clean_macro_f1"]))
print("Kết quả ghi tại reports/corruptions/validation_calibration.json")